# 01 LLM Basics

**Goal for this notebook:** run one OpenAI Responses API call, inspect the response object, and extend it into a minimal multi-turn chatbot.

This notebook focuses on one workflow:

```text
.env / API key -> OpenAI client -> responses.create(...) -> response object -> message history -> chatbot loop
```

## Suggested Pace

1. **Setup check**: verify the Jupyter kernel, API key, and model.
2. **First API call**: run `client.responses.create(...)` and inspect the result.
3. **Core parameters**: understand `model`, `input`, `instructions`, `max_output_tokens`, `output_text`, and `usage`.
4. **Multi-turn context**: maintain message history manually.
5. **Minimal chatbot loop**: wrap the pattern in a `turn(...)` function.

## What Is Optional

The appendix keeps useful but non-essential material for later review:

- Chat Completions and Anthropic API comparison
- JSON mode and structured output
- Streaming
- Model fallback
- Response cache
- OpenAI-compatible model providers

## Required Environment Variables

- Main path: `OPENAI_API_KEY`
- Optional appendix: `ANTHROPIC_API_KEY`
- Main API used here: OpenAI **Responses API**, via `client.responses.create(...)`

## References

- [OpenAI Responses API reference](https://developers.openai.com/api/reference/responses/overview)
- [OpenAI text generation guide](https://platform.openai.com/docs/guides/text)
- [OpenAI migration guide: Chat Completions to Responses](https://platform.openai.com/docs/guides/migrate-to-responses)
- [OpenAI Model Spec overview](https://model-spec.openai.com/2025-12-18.html#overview)
- [Anthropic Messages create API](https://platform.claude.com/docs/en/api/python/messages/create)
- [Jupyter kernels documentation](https://docs.jupyter.org/en/latest/projects/kernels.html)

## Setup Check

Run the next cell first. It loads `.env`, selects the model, and verifies that the notebook can see your API keys.

提示：如果 key 显示为 `False`，优先检查 `.env` 文件位置和当前 notebook kernel。

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()

# Avoid letting an empty OPENAI_BASE_URL override the SDK default endpoint.
if os.getenv("OPENAI_BASE_URL") == "":
    os.environ.pop("OPENAI_BASE_URL", None)

# Shared model setting for this notebook. You can override it in .env with LLM_MODEL.
# If your account does not have access, set LLM_MODEL to another available model.
MODEL = os.getenv("LLM_MODEL", "gpt-5.4-mini")

# Verify that keys are visible to this kernel.
print("OPENAI_KEY    set:", bool(os.getenv("OPENAI_API_KEY")))
print("ANTHROPIC_KEY set:", bool(os.getenv("ANTHROPIC_API_KEY")))
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL") or "<default OpenAI endpoint>")
print("MODEL:", MODEL)

## Section 1 · First OpenAI API Call

A call through the OpenAI Python SDK is still just an HTTPS request with a friendlier Python interface.

提示：SDK 不是魔法，它只是帮你把 Python 代码变成一次 API 请求。

1. Create a client: `client = OpenAI()`.
2. The SDK reads `OPENAI_API_KEY` from the environment.
3. Call an endpoint: `client.responses.create(...)`.
4. The SDK serializes Python values into JSON.
5. It sends a `POST` request to `/v1/responses`.
6. OpenAI returns a JSON response.
7. The SDK turns that JSON into a Python response object.

When debugging an LLM API call, check four things first:

- **Endpoint**: is the request going to the expected API URL?
- **Auth**: is the API key available, valid, and funded?
- **Input**: are `model`, `input`, `instructions`, and parameters valid?
- **Output**: what do `status`, `output_text`, `usage`, and the error message say?

The main path uses the **Responses API** only. Other API shapes are kept in the appendix.

### What Is the Responses API?

`responses.create(...)` is OpenAI's main API for new applications. It can represent text generation, multimodal input, tool calls, structured output, reasoning, and conversation state in one response object.

A minimal call needs two key inputs:

- `model`: which model to use.
- `input`: what to send to the model. This can be a string or a list of messages.

The most useful response fields for beginners are:

- `response.output_text`: the final text, already collected by the SDK.
- `response.output`: the raw structured output list.
- `response.status`: whether the response completed or stopped early.
- `response.usage`: token usage, which matters for cost and context size.

In [ ]:
from openai import OpenAI
client = OpenAI()  # Reads OPENAI_API_KEY by default

response = client.responses.create(
    model=MODEL,
    input="Explain Python decorators in one sentence.",
#     instructions="You are a concise coding tutor. Answer in Chinese.",
#     max_output_tokens=256,
)

# Show the full response object as formatted JSON.
print(json.dumps(response.model_dump(), ensure_ascii=False, indent=2, default=str))

In [ ]:
print("Output text :", response.output_text)
print("Status      :", response.status)
print("Usage       :", response.usage)
print("Model       :", response.model)
print("Response ID :", response.id)

### What Did the API Call Send and Return?

The first code cell sends the request and displays the full `response` object as structured data. The `instructions` parameter asks the model to answer in Chinese, so you can see how behavior changes without changing the user `input`. The next cell extracts the fields you will usually inspect while debugging.

```python
client.responses.create(
    model=MODEL,
    input="...",
    instructions="...",
    max_output_tokens=256,
)
```

Start with these four inputs:

- `model`: selects the model. It affects capability, speed, price, and context window.
- `input`: the content sent to the model. For now, treat it as the user's message.
- `instructions`: higher-priority behavior guidance, such as role, language, and style.
- `max_output_tokens`: the output budget. Too low may truncate the answer; too high raises the maximum possible cost.

And these four outputs:

- `output_text`: the final text answer.
- `status`: whether the response completed.
- `usage`: token usage for cost and context tracking.
- `id`: the unique response ID, useful for linking turns later.

提示：先把一次调用看成“输入参数进去，response object 出来”。后面所有 agent/工具调用/多轮对话，都是在这个基础上扩展。

---

## Optional Sidebar A · Chat Completions and Anthropic

You can skip this sidebar on a first pass and continue with Section 2.

`chat.completions.create(...)` is the older OpenAI chat API. Its main input is `messages`, and the text output is usually read from `resp.choices[0].message.content`.

It is still supported, and many OpenAI-compatible providers use this shape. This notebook focuses on the Responses API because it provides a more unified object model for tools, multimodal input, reasoning, and conversation state.

The next two cells mirror the earlier Responses API pattern: first inspect the full `chat_resp` object, then print the fields you usually need.

Reference: [OpenAI Chat Completions overview](https://developers.openai.com/api/reference/chat-completions/overview)

In [ ]:
chat_resp = client.chat.completions.create(
    model="gpt-5.4-mini",  # Common model for Chat Completions examples
    messages=[
        {"role": "system", "content": "You are a concise coding tutor. Answer in Chinese."},
        {"role": "user", "content": "Explain Python decorators in one sentence."},
    ],
    max_completion_tokens=256,
)

# Show the full Chat Completions response object as formatted JSON.
print(json.dumps(chat_resp.model_dump(), ensure_ascii=False, indent=2, default=str))

In [ ]:
print("Content     :", chat_resp.choices[0].message.content)
print("Finish      :", chat_resp.choices[0].finish_reason)
print("Usage       :", chat_resp.usage)
print("Model       :", chat_resp.model)

### Responses vs Chat Completions

| Topic | Responses API | Chat Completions API |
| --- | --- | --- |
| Method | `client.responses.create(...)` | `client.chat.completions.create(...)` |
| Endpoint | `/v1/responses` | `/v1/chat/completions` |
| Single-turn input | `input="..."` | `messages=[{"role":"user",...}]` |
| Multi-turn input | `input=[...]` or `previous_response_id` | Send the full `messages` list each time |
| System-level guidance | `instructions="..."` or message roles | `system` role inside `messages` |
| Text output | `response.output_text` | `resp.choices[0].message.content` |
| Truncation signal | `status` + `incomplete_details` | `finish_reason == "length"` |
| Structured output | `text={"format": {"type":"json_schema", ...}}` | `response_format={...}` |
| Tools / multimodal | Unified response items | Supported, but represented differently |
| New project default | Recommended here | Useful for legacy code and compatible providers |

Use this table as a reading aid when you see older examples or third-party provider documentation.

### Anthropic Messages API

Anthropic's Messages API has a similar idea but a different shape: `system` is a top-level parameter, `messages` contains user/assistant turns, and `max_tokens` is required.

The next two cells follow the same pattern: first inspect the full `anthropic_resp` object, then print the fields you usually need.

Reference: [Anthropic Messages create API](https://platform.claude.com/docs/en/api/python/messages/create)

In [ ]:
from anthropic import Anthropic
aclient = Anthropic()

anthropic_resp = aclient.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,                     # Required by Anthropic
    system="You are a concise tutor, respond in chinese.",  # Anthropic uses a top-level system parameter
    messages=[{"role": "user", "content": "Explain Python decorators in one sentence."}],
)

# Show the full Anthropic response object as formatted JSON.
print(json.dumps(anthropic_resp.model_dump(), ensure_ascii=False, indent=2, default=str))

In [ ]:
if "anthropic_resp" in globals():
    print("Content     :", anthropic_resp.content[0].text)
    print("Stop reason :", anthropic_resp.stop_reason)
    print("Usage       :", anthropic_resp.usage)
    print("Model       :", anthropic_resp.model)
    print("Message ID  :", anthropic_resp.id)

### Optional Summary · Three API Shapes

You may see three common API shapes in real projects:

1. **OpenAI Responses API**: `client.responses.create(model=..., input=..., instructions=...)`
2. **OpenAI Chat Completions API**: `client.chat.completions.create(model=..., messages=...)`
3. **Anthropic Messages API**: `client.messages.create(model=..., system=..., messages=..., max_tokens=...)`

For this notebook, focus on the first one. The other two are included so you can recognize older code and provider-specific documentation.

---

## Section 2 · Key Parameters and Debugging Signals

This section focuses on a small set of parameters that have an immediate effect on model behavior, plus the response fields that help you debug when something looks wrong.

Start with these common questions:

- **How do I control randomness?** Check `temperature`.
- **Why did the answer stop too early?** Check `max_output_tokens`, `status`, and `incomplete_details`.
- **How expensive or long was this call?** Check `usage`.

Useful rule: **debug an LLM call by checking the input, parameters, and response object.**

提示：第一节课不用背参数表，先学会看到问题时该检查哪里。

> Full parameter tables, JSON mode, streaming, and tools are later-course topics.

In [ ]:
prompt = "Write one sentence about autumn."

for t in [0.0, 0.7, 1.3]:
    print(f"\n--- temperature = {t} ---")
    for i in range(3):
        r = client.responses.create(
            model=MODEL,
            input=prompt,
            instructions="Answer in English. Write only one sentence.",
            temperature=t,
            max_output_tokens=80,
        )
        print(f"  [{i}]", r.output_text)

### How to Read `temperature`

`temperature` controls sampling randomness:

- `0`: more stable and more repeatable.
- `0.7`: common for chatbot-style answers.
- `1.0+`: more varied, useful for creative tasks, but less predictable.

Practical defaults:

- Chatbot: `0.5`-`0.8`
- Structured output / classification / routing: `0`-`0.3`
- Creative writing: `0.9+`

In [ ]:
r = client.responses.create(
    model=MODEL,
    input="Write a 500-word essay about cats.",
    max_output_tokens=50,   # Intentionally too small
)

print("Output:", r.output_text)
print("Status:", r.status)
print("Incomplete details:", r.incomplete_details)
print("Usage:", r.usage)

### How to Read `max_output_tokens`

`max_output_tokens` is the output budget. It limits how many tokens the model may generate.

If the limit is too small, the Responses API may return:

- `status == "incomplete"`
- `incomplete_details.reason == "max_output_tokens"`

Common fixes:

- Increase `max_output_tokens`.
- Split the task into smaller chunks.
- Ask for an outline first, then expand section by section.
- Summarize long conversations before continuing.

In [ ]:
# Optional preview: structured output will be covered in Lesson 02.
# JSON mode requires the word "json" to appear in the input as well.
r = client.responses.create(
    model=MODEL,
    instructions="Return JSON only. The JSON must have keys: title, mood.",
    input="Return a JSON object for a short poem about autumn.",
    text={"format": {"type": "json_object"}},
    temperature=0.3,
)

parsed = json.loads(r.output_text)
print(parsed)

### Optional Preview · `text.format`

You can skip this preview on a first pass and continue with Section 3.

The Responses API can use `text={"format": ...}` to control the output format.

Common options:

- `{"type": "text"}`: default free-form text.
- `{"type": "json_object"}`: JSON mode. It asks for valid JSON but does not guarantee every field is present. The request must mention `json` in the input.
- `{"type": "json_schema", "name": ..., "schema": ..., "strict": True}`: strict structured output, useful for production systems.

提示：JSON mode 只是保证输出是合法 JSON，不保证字段一定完整；生产里更推荐 `json_schema`。

Structured output will be covered in a later lesson with JSON schema and Pydantic.

In [ ]:
first = client.responses.create(
    model=MODEL,
    input="I am learning Python decorators. Explain them in one sentence.",
    instructions="You are a concise coding tutor. Answer in English.",
)

second = client.responses.create(
    model=MODEL,
    input="Based on the previous answer, give a minimal code example.",
    previous_response_id=first.id,
    instructions="You are a concise coding tutor. Answer in English.",
)

print("--- first ---")
print(first.output_text)
print("\n--- second ---")
print(second.output_text)

### How to Read `previous_response_id`

`previous_response_id` is one quick way to connect two Responses API calls:

```python
second = client.responses.create(
    model=MODEL,
    input="Continue the explanation",
    previous_response_id=first.id,
)
```

It means: continue from a previous response.

The deeper idea is more important: **multi-turn chat requires state**. That state can live in different places:

- Manual message history: state is in your Python list or database.
- `previous_response_id`: state is linked through OpenAI response IDs.
- Conversation objects: state is represented through a more explicit conversation abstraction.

Next, use the most transparent approach: manually maintain message history.

## Section 3 · Messages and Multi-Turn Chat

Even with the Responses API, messages are still important because `input` can be a list of message objects.

Common roles:

- `system`: high-level behavior constraints. Many APIs still support it.
- `developer`: application or developer instructions. Common in the Responses API.
- `user`: user input.
- `assistant`: previous model replies.
- `tool`: tool execution results, usually sent after a tool call.

**Messages are context.** Any memory must enter the context somehow: message history, previous response IDs, conversation objects, summary memory, or retrieved external documents.

提示：模型本身不会自动记住上一轮；你要把“需要它记住的内容”重新放进上下文。

In [ ]:
messages = [
    {"role": "developer", "content": "You are a concise coding tutor. Answer in English."},
    {"role": "user", "content": "What is a Python decorator?"},
]

r = client.responses.create(model=MODEL, input=messages)
ans = r.output_text

messages.append({"role": "assistant", "content": ans})
messages.append({"role": "user", "content": "Give the simplest possible example."})

print(">>> messages before sending:")
print(json.dumps(messages, ensure_ascii=False, indent=2))

r2 = client.responses.create(model=MODEL, input=messages)
print("\n--- answer 2 ---\n", r2.output_text)

### Key Idea: Manual History

The previous cell demonstrates client-side short-term memory:

1. First request: send `developer` + `user`.
2. Save the model's reply as an `assistant` message.
3. Second request: send the full message list again.

This is the same mental model used by Chat Completions, but the endpoint is now the Responses API.

Benefit: you fully control the context, which makes it easier to store, trim, summarize, or audit.

Tradeoff: each turn resends more tokens, so long conversations cost more.

提示：这是最重要的工程习惯之一：对话历史是应用层自己管理的状态，不是“模型天然记忆”。

In [ ]:
SYSTEM = "You are a concise coding tutor. Answer in English."
conversation = [{"role": "developer", "content": SYSTEM}]

TOKEN_COLOR = "\033[1;33m"  # bold yellow
RESET_COLOR = "\033[0m"

def turn(user_input: str):
    conversation.append({"role": "user", "content": user_input})
    r = client.responses.create(
        model=MODEL,
        input=conversation,
        temperature=0.7,
    )
    ans = r.output_text
    conversation.append({"role": "assistant", "content": ans})
    print(f"You: {user_input}\nBot: {ans}")
    print(f"{TOKEN_COLOR}  [tokens] {r.usage}{RESET_COLOR}\n")

turn("What is a Python decorator?")
turn("Can you give a logging example?")
turn("What should I add if I want to preserve the function signature?")

In [ ]:
USER_COLOR = "\033[1;36m"  # bold cyan
BOT_COLOR = "\033[1;32m"   # bold green
RESET_COLOR = "\033[0m"

def stateless_turn(user_input: str):
    r = client.responses.create(
        model=MODEL,
        input=user_input,   # Intentionally send only the current turn
    )
    print(f"{USER_COLOR}You: {user_input}{RESET_COLOR}")
    print(f"{BOT_COLOR}Bot: {r.output_text}{RESET_COLOR}\n")

stateless_turn("What did I just ask?")       # The bot has no prior context
stateless_turn("Explain the previous question again.")  # The bot does not know the previous question

---

## Checkpoint · Core Path Complete

At this point, you have completed the main notebook path:

- You ran one OpenAI Responses API call.
- You inspected `output_text`, `status`, `usage`, and `id`.
- You saw that multi-turn chat needs explicit state.
- You built a minimal `turn(...)` chatbot loop.

## Light Assignment

Run this notebook from top to bottom and make sure the main path works:

1. The setup check can read your API keys.
2. The first OpenAI Responses API call returns a response.
3. The key parameter demos run successfully.
4. The multi-turn chatbot demo keeps context across turns.

Everything below is optional appendix material.

---

## Optional Appendix B · Streaming

Streaming lets the user see output earlier, which improves perceived latency. It does not reduce token usage.

The Responses API streaming mode returns a sequence of events. For a basic text stream, the most useful event is `response.output_text.delta`, which contains the next piece of generated text.

In [ ]:
stream = client.responses.create(
    model=MODEL,
    input="Count to 30.",
    stream=True,
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)

## Appendix Summary

Streaming is not required for the core path. The main takeaway from this notebook is: **understand the shape of one LLM API call, and understand why multi-turn chat needs message history.**

## Congratulations！ First Class Complete！

You have finished the first notebook.

In this lesson, you learned how to:

- Load API keys from `.env`
- Make a basic LLM API call
- Inspect the raw response object
- Read useful fields like `output_text`, `status`, `usage`, and `id`
- Control model behavior with key parameters like `instructions`, `temperature`, and `max_output_tokens`
- Understand the difference between stateless calls and multi-turn chat
- Maintain short-term conversation memory with `messages`

提示：后面所有 agent 系统，本质上都会建立在这些基础之上：调用模型、读取 response、控制参数、维护上下文。